# CropSense — EDA and Model Training
This notebook mirrors the full ML training pipeline corresponding to `train.py`.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import shap
import joblib

## 1. Load Data and Exploratory Data Analysis (EDA)

In [ ]:
DATA_PATH = '../backend/data/crop_data.csv'
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes)
print("\nDescribe:\n", df.describe())
print("\nNull Values:\n", df.isnull().sum())

In [ ]:
plt.figure(figsize=(10, 8))
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()

## 2. Preprocessing
Standard scaling for numerical stability and label encoding for the target string categorical values.

In [ ]:
X = df.drop('label', axis=1)
y = df['label']

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 3. Training Classification Models
Training all classifiers and building a soft voting ensemble.

In [ ]:
models = {
    'Naive Bayes': GaussianNB(),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(kernel='rbf', probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=200, learning_rate=0.1, use_label_encoder=False, eval_metric='mlogloss', random_state=42)
}

voting_clf = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=200, random_state=42)),
        ('xgb', XGBClassifier(n_estimators=200, learning_rate=0.1, use_label_encoder=False, eval_metric='mlogloss', random_state=42)),
        ('svm', SVC(kernel='rbf', probability=True, random_state=42))
    ],
    voting='soft'
)
models['Voting Ensemble'] = voting_clf

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    results.append({'Model': name, 'Accuracy': acc, 'F1': f1})
    print(f"{name} - Accuracy: {acc:.4f}, F1: {f1:.4f}")


## 4. Synthesizing Yield and Linear Regression
Since the original dataset lacks yield data, we generate a synthetic target.

In [ ]:
df['synthetic_yield'] = (df['N'] * 0.4) + (df['K'] * 0.3) + (df['rainfall'] * 0.002) + np.random.normal(0, 10, size=len(df))
df['synthetic_yield'] = df['synthetic_yield'].apply(lambda x: max(x, 10))
X_yield = df.drop(['label', 'synthetic_yield'], axis=1)
y_yield = df['synthetic_yield']
X_yield_train, X_yield_test, y_yield_train, y_yield_test = train_test_split(X_yield, y_yield, test_size=0.2, random_state=42)
yield_scaler = StandardScaler()
X_yts = yield_scaler.fit_transform(X_yield_train)
yield_model = LinearRegression()
yield_model.fit(X_yts, y_yield_train)
print("Linear Regression trained for Yield Estimation.")

## 5. K-Means Clustering
For soil profile categorization, based on 5 theoretical profiles.

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans.fit(scaler.transform(X))
print("KMeans Clustering trained.")

## 6. SHAP Value Feature Importance

In [ ]:
explainer = shap.TreeExplainer(models['Random Forest'])
shap_values = explainer.shap_values(X_train_scaled[:100])
shap.summary_plot(shap_values, X_train_scaled[:100], feature_names=X.columns)